# Print fit results table
## Print nice table of fit results and the correlation matrix

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [3]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [4]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/eigen-3.4.0");

In [5]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/ureals.hpp>
#include<uncertainties/math.hpp>

### Load utility functions

In [6]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Open file with fit results

In [7]:
TFile File("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/cisiFit/PreliminaryFitResults.root", "READ");

### Load fit results

In [8]:
std::vector<std::string> *VariableNames = nullptr;
File.GetObject("VariableNames", VariableNames);
std::vector<double> *FitValues = nullptr;
File.GetObject("FitValues", FitValues);
std::vector<double> *FitUncertainties = nullptr;
File.GetObject("FitUncertainties", FitUncertainties);
std::vector<double> *CorrMatrix = nullptr;
File.GetObject("CorrMatrix", CorrMatrix);
std::vector<double> *CovMatrix = nullptr;
File.GetObject("CovMatrix", CovMatrix);

### LaTeX names

In [9]:
std::map<std::string, std::string> LaTeXNames{
    {"BF_KKpipi", "${\\rm BF}(KK\\pi\\pi)$"},
    {"c1", "$c_1$"},
    {"c2", "$c_2$"},
    {"c3", "$c_3$"},
    {"c4", "$c_4$"},
    {"s1", "$s_1$"},
    {"s2", "$s_2$"},
    {"s3", "$s_3$"},
    {"s4", "$s_4$"},
    {"R_M4", "$R_{-4}$"},
    {"R_M3", "$R_{-3}$"},
    {"R_M2", "$R_{-2}$"},
    {"R_M1", "$R_{-1}$"},
    {"R_P1", "$R_1$"},
    {"R_P2", "$R_2$"},
    {"R_P3", "$R_3$"},
    {"BF_KKpipi_KLpipi", "${\\rm BF}(KK\\pi\\pi)_{K_L\\pi\\pi}$"},
    {"rDcosDeltaKpi", "$r_D^{K\\pi}\\cos(\\delta_D^{K\\pi})$"},
    {"rDsinDeltaKpi", "$r_D^{K\\pi}\\sin(\\delta_D^{K\\pi})$"}
};

### $R_i$ model predictions

In [10]:
std::map<std::string, double> Ri_Model{
    {"R_M4", 0.08637600},
    {"R_M3", 0.29746177},
    {"R_M2", 0.39787317},
    {"R_M1", 0.26676530},
    {"R_P1", 0.10997275},
    {"R_P2", 0.40061408},
    {"R_P3", 0.83288628}
};

### Print table

In [11]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 35;
for(std::size_t i = 0; i < VariableNames->size();i++) {
    std::cout << Blank;
    std::cout << std::left << std::setw(Width);
    std::cout << LaTeXNames.at((*VariableNames)[i]) << " & ";
    if((*VariableNames)[i].substr(0, 2) == "BF") {
        std::cout << PrintLaTeXNumber((*FitValues)[i], (*FitUncertainties)[i], -3) << " \\\\" << "\n";
    } else {
        std::cout << PrintLaTeXNumber((*FitValues)[i], (*FitUncertainties)[i]) << " \\\\" << "\n";
    }
    
}

        ${\rm BF}(KK\pi\pi)$                & $(2.869 \pm 0.028)\times 10^{-3}$ \\
        $c_1$                               & $-0.22 \pm 0.08$ \\
        $c_2$                               & $0.79 \pm 0.04$ \\
        $c_3$                               & $0.865 \pm 0.029$ \\
        $c_4$                               & $-0.38 \pm 0.08$ \\
        $s_1$                               & $-0.45 \pm 0.22$ \\
        $s_2$                               & $-0.16 \pm 0.16$ \\
        $s_3$                               & $0.26 \pm 0.14$ \\
        $s_4$                               & $0.47 \pm 0.24$ \\
        $R_{-4}$                            & $0.0876 \pm 0.0027$ \\
        $R_{-3}$                            & $0.294 \pm 0.005$ \\
        $R_{-2}$                            & $0.399 \pm 0.007$ \\
        $R_{-1}$                            & $0.252 \pm 0.007$ \\
        $R_1$                               & $0.119 \pm 0.006$ \\
        $R_2$                               & $0.406 \

### Load bias corrections

In [12]:
std::map<std::string, double> BiasCorrections;
std::ifstream BiasFile("BiasCorrections.txt");
std::string Line;
while(std::getline(BiasFile, Line)) {
    std::stringstream ss(Line);
    std::string Name;
    double Value;
    ss >> Name >> Value;
    BiasCorrections.insert({Name, Value});
}
BiasFile.close();

### Print table after bias corrections

In [13]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 35;
for(std::size_t i = 0; i < VariableNames->size();i++) {
    std::cout << Blank;
    std::cout << std::left << std::setw(Width);
    std::cout << LaTeXNames.at((*VariableNames)[i]) << " & ";
    if((*VariableNames)[i] == "BF_KKpipi_KLpipi") {
        std::cout << PrintLaTeXNumber((*FitValues)[i], (*FitUncertainties)[i], -3) << " \\\\" << "\n";
    } else if((*VariableNames)[i] == "BF_KKpipi") {
        double BiasCorrection = BiasCorrections["BF_KKpipi_PullBias"]*(*FitUncertainties)[i];
        std::cout << PrintLaTeXNumber((*FitValues)[i] - BiasCorrection,
                                      (*FitUncertainties)[i], -3) << " \\\\" << "\n";
    } else if((*VariableNames)[i][0] == 'c' ||
              (*VariableNames)[i][0] == 's' ||
              (*VariableNames)[i].substr(0, 2) == "R_") {
        double BiasCorrection = BiasCorrections[(*VariableNames)[i] + "_PullBias"]*(*FitUncertainties)[i];
        std::cout << PrintLaTeXNumber((*FitValues)[i] - BiasCorrection,
                                      (*FitUncertainties)[i]) << " \\\\" << "\n";
    }/* else if((*VariableNames)[i][0] == 's') {
        double BiasCorrection = BiasCorrections[(*VariableNames)[i] + "_MedianBias"];
        std::cout << "$";
        std::cout << std::fixed << std::setprecision(2) << (*FitValues)[i] - BiasCorrection;
        std::cout << "_{-";
        double MinusError = BiasCorrections[(*VariableNames)[i] + "_MinusPullWidth"]*(*FitUncertainties)[i];
        double PlusError = BiasCorrections[(*VariableNames)[i] + "_PlusPullWidth"]*(*FitUncertainties)[i];
        std::cout << std::fixed << std::setprecision(2) << MinusError;
        std::cout << "}^{+";
        std::cout << std::fixed << std::setprecision(2) << PlusError;
        std::cout << "}$" << " \\\\" << "\n";
    }*/ else {
        std::cout << PrintLaTeXNumber((*FitValues)[i], (*FitUncertainties)[i]) << " \\\\" << "\n";
    }
    
}

        ${\rm BF}(KK\pi\pi)$                & $(2.863 \pm 0.028)\times 10^{-3}$ \\
        $c_1$                               & $-0.22 \pm 0.08$ \\
        $c_2$                               & $0.79 \pm 0.04$ \\
        $c_3$                               & $0.862 \pm 0.029$ \\
        $c_4$                               & $-0.39 \pm 0.08$ \\
        $s_1$                               & $-0.47 \pm 0.22$ \\
        $s_2$                               & $-0.17 \pm 0.16$ \\
        $s_3$                               & $0.26 \pm 0.14$ \\
        $s_4$                               & $0.52 \pm 0.24$ \\
        $R_{-4}$                            & $0.0871 \pm 0.0027$ \\
        $R_{-3}$                            & $0.294 \pm 0.005$ \\
        $R_{-2}$                            & $0.400 \pm 0.007$ \\
        $R_{-1}$                            & $0.252 \pm 0.007$ \\
        $R_1$                               & $0.119 \pm 0.006$ \\
        $R_2$                               & $0.405 \

### Print $R_i$ comparison between fit result and model

In [14]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 10;
for(std::size_t i = 0; i < VariableNames->size();i++) {
    if((*VariableNames)[i].substr(0, 2) != "R_") {
        continue;
    }
    std::cout << Blank;
    std::cout << std::left << std::setw(Width);
    std::cout << LaTeXNames.at((*VariableNames)[i]) << " & ";
    std::cout << PrintLaTeXNumber((*FitValues)[i], (*FitUncertainties)[i]) << " & ";
    std::cout << std::left << std::setw(Width);
    std::stringstream ss;
    ss << std::fixed << std::setprecision(3);
    ss << "$" << Ri_Model.at((*VariableNames)[i]) << "$";
    std::cout << ss.str() << " \\\\" << "\n";
    
}

        $R_{-4}$   & $0.0876 \pm 0.0027$ & $0.086$    \\
        $R_{-3}$   & $0.294 \pm 0.005$ & $0.297$    \\
        $R_{-2}$   & $0.399 \pm 0.007$ & $0.398$    \\
        $R_{-1}$   & $0.252 \pm 0.007$ & $0.267$    \\
        $R_1$      & $0.119 \pm 0.006$ & $0.110$    \\
        $R_2$      & $0.406 \pm 0.012$ & $0.401$    \\
        $R_3$      & $0.815 \pm 0.010$ & $0.833$    \\


### Print correlation matrix

In [15]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 4;
std::ofstream CorrFile("CorrelationMatrix_DataFit.txt");
for(std::size_t i = 0; i < VariableNames->size(); i++) {
    std::cout << Blank;
    for(std::size_t j = 0; j < VariableNames->size(); j++) {
        CorrFile << std::left << std::setw(Width);
        std::stringstream ss;
        ss << "$";
        if((*CorrMatrix)[i + 19*j] > 0.0) {
            ss << "\\phantom{-}";
        }
        ss << std::fixed << std::setprecision(2);
        ss << (*CorrMatrix)[i + 19*j] << "$";
        CorrFile << ss.str();
        if(j == VariableNames->size() - 1) {
            CorrFile << " \\\\" << "\n";
        } else {
            CorrFile << " & ";
        }
    }
}
CorrFile.close();

### Load $c_i$ and $R_i$ to calculate $F_+$

In [16]:
std::vector<double> FitValues_FPlus;
std::vector<std::size_t> Ordering;
for(std::size_t i = 0; i < VariableNames->size();i++) {
    if((*VariableNames)[i][0] == 'c') {
        Ordering.push_back(i);
        double BiasCorrection = BiasCorrections[(*VariableNames)[i] + "_PullBias"]*(*FitUncertainties)[i];
        FitValues_FPlus.push_back((*FitValues)[i] - BiasCorrection);
    }
}
for(std::size_t i = 0; i < VariableNames->size();i++) {
    if((*VariableNames)[i][0] == 'R') {
        Ordering.push_back(i);
        double BiasCorrection = BiasCorrections[(*VariableNames)[i] + "_PullBias"]*(*FitUncertainties)[i];
        FitValues_FPlus.push_back((*FitValues)[i] - BiasCorrection);
    }
}

### Set up covariance matrix to calculate $F_+$

In [25]:
std::vector<double> CovMatrix_flat;
// Set this flag to false to evaluate systematic uncertainty instead of statistical
const bool Statistical = false;
if(Statistical) {
    for(std::size_t i : Ordering) {
        for(std::size_t j : Ordering) {
            CovMatrix_flat.push_back((*CovMatrix)[i + 19*j]);
        }
    }
} else {
    TFile SystCovMatrixFile("../7_SystematicUncertainties/CombinedCovMatrix.root", "READ");
    TMatrixT<double> *SystCovMatrix = nullptr;
    SystCovMatrixFile.GetObject("CovMatrix", SystCovMatrix);
    for(std::size_t i : Ordering) {
        for(std::size_t j : Ordering) {
            CovMatrix_flat.push_back((*SystCovMatrix)(i, j));
        }
    }
    SystCovMatrixFile.Close();
}
std::vector<uncertainties::udouble> x =
    uncertainties::ureals<std::vector<uncertainties::udouble>>(FitValues_FPlus, CovMatrix_flat);

In [26]:
std::vector<uncertainties::udouble> ci_FPlus, Ri_FPlus;
for(std::size_t i = 0; i < 4; i++) {
    ci_FPlus.push_back(x[i]);
}
for(std::size_t i = 0; i < 7; i++) {
    Ri_FPlus.push_back(x[i + 4]);
}
Ri_FPlus.push_back(uncertainties::udouble(1.0, 0.0));

### Function to convert $R_i$ to $K_i$

In [27]:
void ConvertRiToKi(std::vector<uncertainties::udouble> Ri,
                   std::vector<uncertainties::udouble> &Ki,
                   std::vector<uncertainties::udouble> &Kbari) {
    if(Ri.size()%2 == 1) {
        throw std::runtime_error("Ri length must be even");
    }
    if(uncertainties::nom(Ri.back()) != 1.0) {
        throw std::runtime_error("Last element of Ri must be 1.0");
    }
    const std::size_t N = Ri.size()/2;
    std::vector<uncertainties::udouble> MergedKi;
    for(std::size_t i = 0; i < 2*N; i++) {
        MergedKi.push_back(Ri[i]);
        for(std::size_t j = 0; j < i; j++) {
            MergedKi[i] = MergedKi[i]*(1 - Ri[j]);
        }
    }
    for(std::size_t i = 0; i < N; i++) {
        Ki.push_back(MergedKi[i + N]);
        Kbari.push_back(MergedKi[N - i - 1]);
    }
}

In [28]:
std::vector<uncertainties::udouble> Ki_FPlus, Kbari_FPlus;
ConvertRiToKi(Ri_FPlus, Ki_FPlus, Kbari_FPlus);

### Finally calculate $F_+$

In [29]:
uncertainties::udouble FPlus(0.5, 0.0);
for(std::size_t i = 0; i < 4; i++) {
    FPlus += uncertainties::sqrt(Ki_FPlus[i]*Kbari_FPlus[i])*ci_FPlus[i];
    //FPlus += TMath::Sqrt(uncertainties::nom(Ki_FPlus[i]*Kbari_FPlus[i]))*ci_FPlus[i];
}

In [30]:
std::cout << FPlus << "\n";

0.754 ± 0.008
